# Week 5 Breakout Slides: Linear Regression

Weeks ago we let the data organize itself with unsupervised learning. Now we move to **supervised learning**, where every observation comes with a known answer we want to predict. **Linear regression** is the workhorse: it models a continuous target as a weighted sum of the input features, fitting the straight-line relationship that best matches the data. We use it to predict a country's life expectancy from its development indicators, and — just as important — to read off *which* indicators matter and how much.

## Import Libraries

We start with the familiar trio — `numpy`, `pandas`, and `matplotlib`. The `scikit-learn` pieces (the model, the train/test split, and the evaluation metrics) are imported later, each right before we use it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

## Load and Inspect the Data

We reuse the same country development dataset from Week 4 — one row per country, one column per indicator, all numeric.

In [ ]:
path_to_file = Path('your/path')

In [ ]:
df = pd.read_csv(path_to_file / 'country_data.csv')

### Inspect the DataFrame

A quick look confirms the columns and shape before we start modeling.

In [ ]:
df

## Define Features and Target

Unlike the unsupervised work in Week 4, regression is **supervised**: we single out one column to predict (the **target**, `y`) and use the rest as inputs (the **features**, `X`). Here we predict **life expectancy** from a country's other development indicators.

In [ ]:
target = 'life_expec'

X = df.drop(columns=target)
y = df[target]

## Train/Test Split

To judge how well the model generalizes, we hold out part of the data. We fit on the **training set** and evaluate on the **test set** the model never saw — this is what catches overfitting. `random_state` makes the split reproducible.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Fit a Linear Regression Model

`scikit-learn`'s `LinearRegression` finds the coefficients that minimize the sum of squared errors (ordinary least squares). Fitting is a single call.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression().fit(X_train, y_train)

## Evaluate the Model

With a fitted model in hand, we predict on the held-out test set and quantify how close those predictions are to the truth.

### Predictions

`.predict()` returns the model's estimated life expectancy for each country in the test set.

In [ ]:
y_pred = model.predict(X_test)

### Metrics

Three standard regression scores summarize performance:

- **R²** — the fraction of variance in the target the model explains (1.0 is perfect, 0.0 is no better than predicting the mean).
- **MAE** — mean absolute error, the average miss in the target's own units (years).
- **RMSE** — root mean squared error, like MAE but penalizes large misses more heavily.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R^2:  {r2:.3f}")
print(f"MAE:  {mae:.3f} years")
print(f"RMSE: {rmse:.3f} years")

### Predicted vs. Actual

Plotting predictions against the true values is the quickest visual gut-check. Points hugging the dashed 45-degree line mean the model is on target; systematic departures reveal where it struggles.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, color='steelblue', alpha=0.8)

# 45-degree reference line: perfect predictions land here
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', label='Perfect prediction')

ax.set_xlabel('Actual Life Expectancy (years)')
ax.set_ylabel('Predicted Life Expectancy (years)')
ax.set_title('Predicted vs. Actual')
ax.legend()
ax.grid(alpha=0.3)

### Residual Plot

A **residual** is the leftover error (actual minus predicted). Plotting residuals against the predictions should look like a shapeless cloud centered on zero. Curves or a fanning-out pattern signal that the model is missing structure in the data.

In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_pred, residuals, color='steelblue', alpha=0.8)
ax.axhline(0, color='k', linestyle='--')

ax.set_xlabel('Predicted Life Expectancy (years)')
ax.set_ylabel('Residual (actual - predicted)')
ax.set_title('Residual Plot')
ax.grid(alpha=0.3)

## Interpreting the Model: Coefficients

Linear regression is prized for transparency. Each **coefficient** is the expected change in life expectancy for a one-unit increase in that feature, holding the others fixed — positive pushes life expectancy up, negative pulls it down. The intercept is the baseline when all features are zero.

In [ ]:
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"Intercept: {model.intercept_:.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
coefs.plot.barh(color='steelblue', edgecolor='k', ax=ax)
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Coefficient (change in life expectancy per unit)')
ax.set_title('Linear Regression Coefficients')
ax.grid(alpha=0.3)